# PG-LIF — Phase P1: SHD Benchmark (Regime A)
**Project:** PG-LIF · **Phase:** P1 of the Research and Implementation Plan · **Feeds:** manuscript Table 1 (Section 7.1)

Trains PG-LIF and the baselines (LIF, ALIF, TC-LIF, DH-LIF) on the **Spiking Heidelberg Digits** dataset in an **identical network skeleton** (700 → recurrent spiking layer → leaky readout, cross-entropy on summed readout potential), so that differences are attributable to the neuron model. TC-LIF uses the dynamics of the authors' official repository (github.com/ZhangShimin1/TC-LIF); DH-LIF is a re-implementation per Zheng et al. (2024) and must be validated against published numbers before manuscript use.

**Requirements:** GPU runtime (Runtime → Change runtime type → T4). SHD is downloaded once (~460 MB extracted) and cached in `My Drive/PG_LIF/data/SHD/`.

**Modes:** `PAPER_MODE = False` (default) is a quick pass — T = 100 bins, 20 epochs, seed 0 — to verify the pipeline and get a first ranking (~10–20 min per model on a T4). `PAPER_MODE = True` reproduces the official TC-LIF protocol — T = 250, 100 epochs, LR drops at [40, 80], seeds 0–4 — and is what Table 1 requires. Runs are resumable: finished (model, seed) pairs are skipped automatically.

In [ ]:
# --- Setup: Drive, folders ---
import os, json, time, gzip, shutil, urllib.request
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT):    # some environments expose the no-space alias only
        ROOT = '/content/drive/MyDrive'
    BASE = os.path.join(ROOT, 'PG_LIF')
except Exception:
    BASE = './PG_LIF'   # local fallback (development only)
DATA = os.path.join(BASE, 'data', 'SHD')
P1 = os.path.join(BASE, 'P1_results')
os.makedirs(DATA, exist_ok=True); os.makedirs(P1, exist_ok=True)
print('Base folder:', BASE)

In [ ]:
# --- CONFIG ---
PAPER_MODE = False        # True = official TC-LIF SHD protocol (T=250, 100 epochs, seeds 0-4)
MODELS = ['LIF', 'ALIF', 'TCLIF', 'DHLIF', 'PGLIF']   # subset to shorten a session
SEEDS  = [0, 1, 2, 3, 4] if PAPER_MODE else [0]
T_BINS   = 250 if PAPER_MODE else 100
EPOCHS   = 100 if PAPER_MODE else 20
SCHEDULE = [40, 80] if PAPER_MODE else [12]
BATCH  = 64
LR     = 5e-4
HIDDEN = 128
MAX_TIME = 1.4            # seconds of each SHD sample used
N_IN, N_OUT = 700, 20
RUN_TAG = ('paper' if PAPER_MODE else 'quick') + f'_T{T_BINS}_E{EPOCHS}_H{HIDDEN}'
OUT = os.path.join(P1, RUN_TAG); os.makedirs(OUT, exist_ok=True)
with open(os.path.join(OUT, 'config.json'), 'w') as f:
    json.dump({k: v for k, v in globals().items() if k in
               ['PAPER_MODE','MODELS','SEEDS','T_BINS','EPOCHS','SCHEDULE','BATCH','LR','HIDDEN','MAX_TIME']},
              f, indent=2, default=str)
print('Run folder:', OUT)

## Data: download and cache SHD, dense binning
Official files from the Zenke lab (`zenkelab.org/datasets`). Events are binned into `T_BINS` steps over the first `MAX_TIME` seconds (the standard SpyTorch/TC-LIF preprocessing). Dense batches are built on the fly to keep RAM modest.

In [ ]:
import numpy as np, h5py
URLS = {'shd_train.h5': 'https://zenkelab.org/datasets/shd_train.h5.gz',
        'shd_test.h5':  'https://zenkelab.org/datasets/shd_test.h5.gz'}
for name, url in URLS.items():
    dst = os.path.join(DATA, name)
    if not os.path.exists(dst):
        gz = dst + '.gz'
        print('downloading', url)
        urllib.request.urlretrieve(url, gz)
        with gzip.open(gz, 'rb') as fi, open(dst, 'wb') as fo: shutil.copyfileobj(fi, fo)
        os.remove(gz)
    print(name, 'ready,', os.path.getsize(dst)//(1<<20), 'MB')

def load_split(fname):
    with h5py.File(os.path.join(DATA, fname), 'r') as f:
        times = [np.array(t) for t in f['spikes']['times']]
        units = [np.array(u) for u in f['spikes']['units']]
        labels = np.array(f['labels'], dtype=np.int64)
    return times, units, labels

TR = load_split('shd_train.h5'); TE = load_split('shd_test.h5')
print('train samples:', len(TR[2]), '| test samples:', len(TE[2]))

def batches(split, batch_size, shuffle, T=None, device='cpu'):
    import torch
    times, units, labels = split
    T = T or T_BINS
    idx = np.random.permutation(len(labels)) if shuffle else np.arange(len(labels))
    for b0 in range(0, len(idx), batch_size):
        sel = idx[b0:b0+batch_size]
        x = torch.zeros(len(sel), T, N_IN)
        for i, j in enumerate(sel):
            tt = times[j]; uu = units[j]
            keep = tt < MAX_TIME
            tb = np.clip((tt[keep] / MAX_TIME * T).astype(int), 0, T-1)
            x[i, tb, uu[keep]] = 1.0
        yield x.to(device), torch.as_tensor(labels[sel]).to(device)

## Neuron cells
All cells share the triangle surrogate and expose `(spikes, spike_count)` over a `(B, T, N)` input current sequence plus a recurrent weight applied to their own output. Per-neuron hyperparameters follow each model's authors (TC-LIF: threshold 1.5, γ = 0.5, learnable decays initialized at 0; DH-LIF: 4 branches with learnable timing factors spread across timescales); the skeleton, optimizer, schedule, and loss are identical for all.

In [ ]:
import torch, torch.nn as nn, math

class Triangle(torch.autograd.Function):
    gamma = 1.0
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x >= 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * torch.clamp(1.0 - x.abs() / Triangle.gamma, min=0.0)
spike_fn = Triangle.apply

def decay(tau): return math.exp(-1.0 / tau)   # per-step decay, tau in steps

class LIFCell(nn.Module):
    th = 1.0
    def __init__(self, N): super().__init__(); self.N = N; self.am = decay(20)
    def init(self, B, dev): self.v = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v = self.am * self.v + I
        s = spike_fn(self.v - self.th)
        self.v = self.v - s.detach() * self.th          # soft reset
        return s

class ALIFCell(nn.Module):
    th = 1.0; beta = 1.6
    def __init__(self, N): super().__init__(); self.N = N; self.am = decay(20); self.aa = decay(200)
    def init(self, B, dev):
        self.v = torch.zeros(B, self.N, device=dev); self.a = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v = self.am * self.v + I
        th = self.th + self.beta * self.a
        s = spike_fn(self.v - th)
        self.v = self.v - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

class TCLIFCell(nn.Module):
    """Official TC-LIF dynamics (ZhangShimin1/TC-LIF): v1 = v1 - sig(d0)*v2 + I ; v2 = v2 + sig(d1)*v1;
       spike on v2 >= 1.5; soft reset v1 -= gamma*s, v2 -= th*s; d learnable, init 0."""
    th = 1.5; gamma_r = 0.5
    def __init__(self, N):
        super().__init__(); self.N = N
        self.d = nn.Parameter(torch.zeros(2))
    def init(self, B, dev):
        self.v1 = torch.zeros(B, self.N, device=dev); self.v2 = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v1 = self.v1 - torch.sigmoid(self.d[0]) * self.v2 + I
        self.v2 = self.v2 + torch.sigmoid(self.d[1]) * self.v1
        s = spike_fn(self.v2 - self.th)
        self.v1 = self.v1 - s * self.gamma_r
        self.v2 = self.v2 - s * self.th
        return s

class DHLIFCell(nn.Module):
    """Re-implementation per Zheng et al. 2024: k dendritic branches with learnable timing factors;
       input current is split across branches; membrane integrates the summed branch currents."""
    th = 1.0; K = 4
    def __init__(self, N):
        super().__init__(); self.N = N; self.am = decay(20)
        init_a = torch.tensor([0.5, 0.8, 0.95, 0.99])
        logit = torch.log(init_a / (1 - init_a))
        self.branch_logit = nn.Parameter(logit.view(self.K, 1).repeat(1, N))   # (K, N)
        self.mix = nn.Parameter(torch.ones(self.K, N) / self.K)
    def init(self, B, dev):
        self.i = torch.zeros(B, self.K, self.N, device=dev); self.v = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        ad = torch.sigmoid(self.branch_logit)                                   # (K, N)
        self.i = ad.unsqueeze(0) * self.i + I.unsqueeze(1) / self.K
        self.v = self.am * self.v + (self.mix.unsqueeze(0) * self.i).sum(1)
        s = spike_fn(self.v - self.th)
        self.v = self.v - s.detach() * self.th
        return s

class PGLIFCell(nn.Module):
    """PG-LIF (manuscript Eqs. 6-10), input routing handled by the skeleton:
       feedforward current -> dendrite, recurrent current -> soma. Learnable: alpha_p (per neuron), kappa."""
    th = 1.0; th_d = 1.0; P0 = 1.0; beta = 1.0
    def __init__(self, N, tref_p=10):
        super().__init__(); self.N = N
        self.am = decay(20); self.ad = decay(20); self.aa = decay(200)
        ap0 = decay(T_BINS / 2)
        self.ap_logit = nn.Parameter(torch.full((N,), math.log(ap0 / (1 - ap0))))
        self.kappa = nn.Parameter(torch.ones(N))
        self.tref_p = tref_p
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.vd, self.p, self.a = z(), z(), z(), z()
        self.rp = torch.zeros(B, self.N, device=dev)
    def forward(self, I_ff, I_rec):
        self.vd = self.ad * self.vd + I_ff
        ed = spike_fn(self.vd - self.th_d) * (self.rp == 0).float()
        self.rp = torch.clamp(self.rp - 1, min=0) + ed.detach() * self.tref_p
        self.p = torch.sigmoid(self.ap_logit) * self.p + self.P0 * ed
        self.vs = self.am * self.vs + I_rec + self.kappa * self.p * (1 - self.am)
        th = self.th + self.beta * self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

## Network skeleton and training
One recurrent spiking layer (identical for all neurons), leaky readout, cross-entropy on the readout potential summed over time. Adam, StepLR (0.1x at the scheduled epochs), gradient clipping at 5. Spike counts are recorded at every evaluation.

In [ ]:
class RecSNN(nn.Module):
    def __init__(self, cell_name):
        super().__init__()
        self.cell_name = cell_name
        self.w_in = nn.Linear(N_IN, HIDDEN)
        self.w_rec = nn.Linear(HIDDEN, HIDDEN, bias=False)
        self.cell = {'LIF': LIFCell, 'ALIF': ALIFCell, 'TCLIF': TCLIFCell,
                     'DHLIF': DHLIFCell, 'PGLIF': PGLIFCell}[cell_name](HIDDEN)
        self.w_out = nn.Linear(HIDDEN, N_OUT)
        self.a_out = decay(20)
        nn.init.orthogonal_(self.w_rec.weight)
    def forward(self, x):                       # x: (B, T, N_IN)
        B, T, _ = x.shape; dev = x.device
        self.cell.init(B, dev)
        s = torch.zeros(B, HIDDEN, device=dev)
        out = torch.zeros(B, N_OUT, device=dev); vo = torch.zeros(B, N_OUT, device=dev)
        n_spk = 0.0
        for t in range(T):
            iff = self.w_in(x[:, t]); irec = self.w_rec(s)
            s = self.cell(iff, irec) if self.cell_name == 'PGLIF' else self.cell(iff + irec)
            n_spk = n_spk + s.detach().sum()
            vo = self.a_out * vo + self.w_out(s)
            out = out + vo
        return out, n_spk / B

def evaluate(model, device):
    model.eval(); correct = tot = 0; spk = 0.0; nb = 0
    with torch.no_grad():
        for x, y in batches(TE, 256, shuffle=False, device=device):
            out, ns = model(x)
            correct += (out.argmax(1) == y).sum().item(); tot += len(y)
            spk += ns.item(); nb += 1
    return correct / tot, spk / nb

def train_one(model_name, seed, device):
    res_file = os.path.join(OUT, f'{model_name}_s{seed}.json')
    if os.path.exists(res_file):
        print(f'[skip] {model_name} seed {seed} already done'); return json.load(open(res_file))
    torch.manual_seed(seed); np.random.seed(seed)
    model = RecSNN(model_name).to(device)
    n_par = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=SCHEDULE, gamma=0.1)
    crit = nn.CrossEntropyLoss()
    best, best_spk, hist = 0.0, 0.0, []
    for ep in range(EPOCHS):
        model.train(); t0 = time.time()
        for x, y in batches(TR, BATCH, shuffle=True, device=device):
            opt.zero_grad()
            out, _ = model(x)
            loss = crit(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        sch.step()
        acc, spk = evaluate(model, device)
        hist.append({'epoch': ep, 'test_acc': acc, 'spikes': spk})
        if acc > best:
            best, best_spk = acc, spk
            torch.save(model.state_dict(), os.path.join(OUT, f'{model_name}_s{seed}_best.pt'))
        print(f'{model_name} s{seed} ep{ep:03d}  acc {acc:.4f} (best {best:.4f})  '
              f'spk/sample {spk:.0f}  {time.time()-t0:.0f}s')
    res = {'model': model_name, 'seed': seed, 'best_test_acc': best,
           'spikes_per_sample': best_spk, 'params': n_par, 'history': hist}
    json.dump(res, open(res_file, 'w'), indent=2)
    return res

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu': print('WARNING: no GPU detected - this will be very slow. Switch runtime to GPU.')
results = []
for m in MODELS:
    for sd in SEEDS:
        results.append(train_one(m, sd, device))

## Aggregate results and decision gate

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict
agg = defaultdict(list)
for r in results: agg[r['model']].append(r)
rows = []
for m in MODELS:
    accs = np.array([r['best_test_acc'] for r in agg[m]])
    spks = np.array([r['spikes_per_sample'] for r in agg[m]])
    rows.append({'model': m, 'acc_mean': accs.mean(), 'acc_std': accs.std(),
                 'spikes': spks.mean(), 'params': agg[m][0]['params'], 'n_seeds': len(accs)})
    print(f"{m:6s}  acc {accs.mean()*100:.2f} +- {accs.std()*100:.2f} %   "
          f"spikes/sample {spks.mean():.0f}   params {agg[m][0]['params']}")
json.dump(rows, open(os.path.join(OUT, 'aggregate.json'), 'w'), indent=2)

plt.figure(figsize=(7, 4))
plt.bar([r['model'] for r in rows], [r['acc_mean']*100 for r in rows],
        yerr=[r['acc_std']*100 for r in rows], capsize=4)
plt.ylabel('SHD test accuracy (%)'); plt.title(f'P1 ({RUN_TAG})')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig_P1_accuracy.png'), dpi=300); plt.show()

pg = next((r for r in rows if r['model'] == 'PGLIF'), None)
tc = next((r for r in rows if r['model'] == 'TCLIF'), None)
dh = next((r for r in rows if r['model'] == 'DHLIF'), None)
if pg and (tc or dh):
    ref = max([r for r in (tc, dh) if r], key=lambda r: r['acc_mean'])
    gap = (pg['acc_mean'] - ref['acc_mean']) * 100
    noise = 2 * max(pg['acc_std'], ref['acc_std']) * 100 if len(SEEDS) > 1 else 1.5
    verdict = ('PASSED - PG-LIF within noise of the strongest two-compartment baseline or better; proceed to P2'
               if gap >= -noise else
               'NOT MET - PG-LIF clearly below the strongest baseline; consult fallback in plan Sec. 7 before P2')
    print(f'\nDECISION GATE P1: gap to {ref["model"]} = {gap:+.2f} pp (noise band {noise:.2f} pp) -> {verdict}')
if not PAPER_MODE:
    print('\nNOTE: quick mode. Table 1 of the manuscript requires PAPER_MODE = True (T=250, 100 epochs, seeds 0-4).')

## Appendix: revised phase diagram for the manuscript (P0 refinement)
The P0 run showed that under *constant* dendritic drive the ISI-CV criterion never labels the bursting regime. The manuscript figure should instead use the pulsed-event protocol of Experiment 3: dendritic events every 500 ms, classification by episode structure — silent, tonic (firing independent of events), plateau-driven (firing episodes locked to events), and adaptive-burst (short truncated episodes). This cell produces `fig_E5_revised.png` for Section 7.4 and saves it in this run's folder.

In [ ]:
K = 15; Tpd = 1600
kaps = torch.linspace(0.0, 3.0, K); betas = torch.linspace(0.0, 2.0, K)
KK, BB = torch.meshgrid(kaps, betas, indexing='ij')
NN = K * K
# forward-only PG-LIF grid (float32, no reset of manuscript semantics needed beyond P0 class)
am, adk, apd, aa = decay(20), decay(20), decay(200), decay(200)
vs = torch.zeros(NN); vd = torch.zeros(NN); p = torch.zeros(NN); a = torch.zeros(NN)
rp = torch.zeros(NN); spikes = torch.zeros(Tpd, NN)
kap = KK.flatten(); bet = BB.flatten()
event_times = [100, 600, 1100]
for t in range(Tpd):
    Id = 25.0 if t in event_times else 0.0
    vd = adk * vd + (1 - adk) * Id
    ed = ((vd >= 1.0) & (rp == 0)).float()
    rp = torch.clamp(rp - 1, min=0) + ed * 30
    p = apd * p + ed
    vs = am * vs + (1 - am) * (0.6 + kap * p)
    s = (vs >= 1.0 + bet * a).float()
    vs = torch.where(s.bool(), torch.zeros_like(vs), vs)
    a = aa * a + s
    spikes[t] = s
cls = torch.zeros(NN)
for i in range(NN):
    st = torch.nonzero(spikes[:, i]).flatten()
    if len(st) == 0: cls[i] = 0; continue
    pre = (st < event_times[0]).sum().item()          # spikes before any event => tonic-like
    if pre > 0: cls[i] = 1; continue
    # spikes per episode (windows after each event)
    per_ep = [((st >= t0) & (st < t0 + 480)).sum().item() for t0 in event_times]
    cls[i] = 3 if np.mean(per_ep) >= 5 else 2          # 3 plateau-driven, 2 adaptive-burst (truncated)
plt.figure(figsize=(6, 4.5))
im = plt.imshow(cls.reshape(K, K).T.numpy(), origin='lower', aspect='auto', cmap='viridis',
                extent=[float(kaps[0]), float(kaps[-1]), float(betas[0]), float(betas[-1])])
plt.xlabel('kappa'); plt.ylabel('beta')
plt.title('Regimes, pulsed events (0 silent, 1 tonic, 2 adaptive-burst, 3 plateau-driven)')
plt.colorbar(im); plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig_E5_revised.png'), dpi=300); plt.show()
print('Saved revised phase diagram for manuscript Sec. 7.4.')

### Next steps
1. If the quick pass ranks sensibly, set `PAPER_MODE = True` and rerun (long session; runs resume across disconnects because finished pairs are skipped).
2. Insert the aggregate table into manuscript Table 1 and update the abstract's `[[RESULT]]` markers.
3. **Phase P2 (before SSC/PS-MNIST):** ablation (a) — replace the plateau with a second ALIF adaptation variable of equal time constant inside `PGLIFCell`. This is the make-or-break diagnostic of the plan; I will provide it as a small variant cell on request.
4. Before manuscript use, validate the DH-LIF re-implementation by reproducing a published DH-LIF SHD number within tolerance.